# 게임코드

In [1]:
import random
import math

# ----- 확률 파라미터 -----
p = 1 / 125
q = 1 - p

total = int(input("얼마를 충전하시겠습니까? "))
jackpot_num_list = []
previous_betting = 0

# 구간 정의(양끝 포함)
intervals = [(1, 28), (29, 64), (65, 115), (116, 200), (201, float('inf'))]


# ------------------------------
# 임베딩 벡터 관련 함수들
# ------------------------------

def make_jackpot_num():
    """기하분포(p=1/125)를 시뮬레이션해서 첫 잭팟 시점을 반환"""
    jackpot_num = 1
    while True:
        if random.randint(1, 125) == 1:
            return jackpot_num
        jackpot_num += 1

def classify_interval(x, intervals):
    """값 x가 속한 구간의 인덱스(0~len-1)를 반환"""
    for i, (lo, hi) in enumerate(intervals):
        if lo <= x <= hi:
            return i
    return None

def signed_streaks_by_interval(nums, intervals):
    """각 구간의 '부호 있는 스트릭' 계산"""
    seq = [classify_interval(x, intervals) for x in nums]
    m = len(intervals)
    L = len(seq)
    if L == 0:
        return [0] * m

    streaks = []
    for target in range(m):
        if seq[-1] == target:
            r = 0
            j = L - 1
            while j >= 0 and seq[j] == target:
                r += 1
                j -= 1
            streaks.append(+r)
        else:
            g = 0
            j = L - 1
            while j >= 0 and seq[j] != target:
                g += 1
                j -= 1
            streaks.append(-g)
    return streaks

def count_all_intervals(nums, intervals):
    """전체 잭팟 기록에서 각 구간의 누적 출현 횟수 계산"""
    counts = [0] * len(intervals)
    for x in nums:
        idx = classify_interval(x, intervals)
        if idx is not None:
            counts[idx] += 1
    return counts, len(nums)

def get_disappeared_bin(nums, intervals, window_size=5):
    """최근 window_size개 구간 중 사라진 구간 계산"""
    if len(nums) < window_size:
        return None
    old_value = nums[-window_size]
    disappeared_bin = classify_interval(old_value, intervals)
    return disappeared_bin

def moving_average(data, n):
    """최근 n개 평균"""
    if len(data) == 0:
        return 0
    subset = data[-n:] if len(data) >= n else data
    return round(sum(subset) / len(subset))

def interval_distribution(nums, intervals, n):
    """최근 n개 구간 분포"""
    recent = nums[-n:]
    counts = [0] * len(intervals)
    for num in recent:
        idx = classify_interval(num, intervals)
        if idx is not None:
            counts[idx] += 1
    return counts

# ------------------------------
# 30차원 임베딩 벡터 생성 함수
# ------------------------------

def create_embedding_vector(jackpot_num_list, intervals, play_num):
    """
    의미 기반 30차원 임베딩 벡터 생성 (모든 값 소수 둘째 자리 반올림)
    """

    # 1. 최근 구간 분포(5,10,15,20)
    interval_features = []
    for n in [5, 10, 15, 20]:
        interval_features.extend(interval_distribution(jackpot_num_list, intervals, n))

    # 2. 최근 평균 (5, 10, 15, 20)
    avg_features = [moving_average(jackpot_num_list, n) for n in [5, 10, 15, 20]]

    # 3. 구간별 연속정도 (signed streak)
    streak_features = signed_streaks_by_interval(jackpot_num_list, intervals)

    # 4. 당첨 확률 (기하 CDF)
    p = 1/125
    q = 1 - p
    probability_cdf = (1 - (q ** play_num))

    # --- 하나의 임베딩 벡터로 합치기 ---
    embedding_vector = []
    embedding_vector.extend(interval_features)
    embedding_vector.extend(avg_features)
    embedding_vector.extend(streak_features)
    embedding_vector.append(probability_cdf)

    # 🔥 모든 요소 소수 둘째 자리 반올림
    embedding_vector = [round(float(x), 2) for x in embedding_vector]

    return embedding_vector


# ------------------------------
# 메인 게임 함수
# ------------------------------

def slot_machine():
    global total, previous_betting, jackpot_num_list
    jackpot_num = make_jackpot_num()
    play_num = 0

    while True:
        betting_str = input("얼마를 배팅 하시겠습니까? ")

        if betting_str == '':
            if previous_betting == 0:
                print("이전 배팅 금액이 없습니다. 배팅 금액을 입력해주세요.")
                continue
            else:
                betting = previous_betting
        else:
            try:
                betting = int(betting_str)
                if betting <= 0:
                    print("배팅 금액은 0보다 커야 합니다.")
                    continue
            except ValueError:
                print("유효한 배팅 금액을 입력해주세요.")
                continue

        if betting > total:
            print("배팅 금액이 잔액을 초과합니다.")
            continue

        previous_betting = betting
        play_num += 1
        total -= betting

        probability = (1 - (q ** play_num)) * 100

        # 💡 구간분포: 5, 10, 15, 20개 기준으로 계산
        interval_counts = {n: interval_distribution(jackpot_num_list, intervals, n) for n in [5, 10, 15, 20]}

        # 💡 이동평균: 5, 10, 15, 20개 기준으로 계산
        averages = {n: moving_average(jackpot_num_list, n) for n in [5, 10, 15, 20]}

        signed_streaks = signed_streaks_by_interval(jackpot_num_list, intervals)
        total_counts, n_total = count_all_intervals(jackpot_num_list, intervals)

        disappeared_bin = get_disappeared_bin(jackpot_num_list, intervals, window_size=5)

        print("\n")
        print(f"현재 {play_num}회 배팅 중입니다. total: {total}")
        for n in [5, 10, 15, 20]:
            print(f"구간분포(최근{n}) {interval_counts[n]}")
        print(f"당첨확률(기하CDF): {probability:.2f}%")

        avg_str = ", ".join([f"{n}개={averages[n]}" for n in [5, 10, 15, 20]])
        print(f"최근 평균: {avg_str}")
        print(f"구간별 연속정도 : {signed_streaks}")
        print(f"전체 {n_total}개 구간별 누적 횟수: {total_counts}")

        if disappeared_bin is not None:
            lo, hi = intervals[disappeared_bin]
            print(f"최근 5회 기준 사라진 구간: {disappeared_bin + 1} ({lo}~{hi})")
        else:
            print(f"최근 5회 기준 사라진 구간: 없음 (데이터 부족)")

        print(f"(1,28)(29,64)(65,115)(116,200)(201), 이전 {jackpot_num_list[-5:]}")

        # -------------------------
        # 📌 임베딩 벡터 출력 추가됨!
        # -------------------------
        emb = create_embedding_vector(jackpot_num_list, intervals, play_num)
        print("\n📌 생성된 임베딩벡터 (길이:", len(emb), ")")
        print(emb)
        # -------------------------

        # 🎰 잭팟 처리
        if play_num == jackpot_num:
            total += betting * 110
            jackpot_num_list.append(play_num)
            print(f"이야~ 대박!🧞‍♂️ 축하합니다! 잭팟입니다! total: {total}")
            print("🧚‍♂️ 🧚‍♀️")
            break


# ------------------------------
# 게임 반복 실행
# ------------------------------

while True:
    slot_machine()


얼마를 충전하시겠습니까?  50000
얼마를 배팅 하시겠습니까?  50




현재 1회 배팅 중입니다. total: 49950
구간분포(최근5) [0, 0, 0, 0, 0]
구간분포(최근10) [0, 0, 0, 0, 0]
구간분포(최근15) [0, 0, 0, 0, 0]
구간분포(최근20) [0, 0, 0, 0, 0]
당첨확률(기하CDF): 0.80%
최근 평균: 5개=0, 10개=0, 15개=0, 20개=0
구간별 연속정도 : [0, 0, 0, 0, 0]
전체 0개 구간별 누적 횟수: [0, 0, 0, 0, 0]
최근 5회 기준 사라진 구간: 없음 (데이터 부족)
(1,28)(29,64)(65,115)(116,200)(201), 이전 []

📌 생성된 임베딩벡터 (길이: 30 )
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01]


얼마를 배팅 하시겠습니까?  




현재 2회 배팅 중입니다. total: 49900
구간분포(최근5) [0, 0, 0, 0, 0]
구간분포(최근10) [0, 0, 0, 0, 0]
구간분포(최근15) [0, 0, 0, 0, 0]
구간분포(최근20) [0, 0, 0, 0, 0]
당첨확률(기하CDF): 1.59%
최근 평균: 5개=0, 10개=0, 15개=0, 20개=0
구간별 연속정도 : [0, 0, 0, 0, 0]
전체 0개 구간별 누적 횟수: [0, 0, 0, 0, 0]
최근 5회 기준 사라진 구간: 없음 (데이터 부족)
(1,28)(29,64)(65,115)(116,200)(201), 이전 []

📌 생성된 임베딩벡터 (길이: 30 )
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.02]


얼마를 배팅 하시겠습니까?  


...

얼마를 배팅 하시겠습니까?  




현재 24회 배팅 중입니다. total: 53300
구간분포(최근5) [0, 0, 1, 1, 0]
구간분포(최근10) [0, 0, 1, 1, 0]
구간분포(최근15) [0, 0, 1, 1, 0]
구간분포(최근20) [0, 0, 1, 1, 0]
당첨확률(기하CDF): 17.53%
최근 평균: 5개=143, 10개=143, 15개=143, 20개=143
구간별 연속정도 : [-2, -2, 1, -1, -2]
전체 2개 구간별 누적 횟수: [0, 0, 1, 1, 0]
최근 5회 기준 사라진 구간: 없음 (데이터 부족)
(1,28)(29,64)(65,115)(116,200)(201), 이전 [199, 87]

📌 생성된 임베딩벡터 (길이: 30 )
[0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 143.0, 143.0, 143.0, 143.0, -2.0, -2.0, 1.0, -1.0, -2.0, 0.18]


얼마를 배팅 하시겠습니까?  




현재 25회 배팅 중입니다. total: 53100
구간분포(최근5) [0, 0, 1, 1, 0]
구간분포(최근10) [0, 0, 1, 1, 0]
구간분포(최근15) [0, 0, 1, 1, 0]
구간분포(최근20) [0, 0, 1, 1, 0]
당첨확률(기하CDF): 18.19%
최근 평균: 5개=143, 10개=143, 15개=143, 20개=143
구간별 연속정도 : [-2, -2, 1, -1, -2]
전체 2개 구간별 누적 횟수: [0, 0, 1, 1, 0]
최근 5회 기준 사라진 구간: 없음 (데이터 부족)
(1,28)(29,64)(65,115)(116,200)(201), 이전 [199, 87]

📌 생성된 임베딩벡터 (길이: 30 )
[0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 143.0, 143.0, 143.0, 143.0, -2.0, -2.0, 1.0, -1.0, -2.0, 0.18]


KeyboardInterrupt: Interrupted by user

# 임베딩 벡터 생성

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import numpy as np

# =========================
# 1. 하이퍼파라미터 설정
# =========================
INPUT_DIM = 14
OUTPUT_DIM = 5
HIDDEN_DIM = 64
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 2. MLP 모델 정의
# =========================
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(INPUT_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.BatchNorm1d(HIDDEN_DIM),

            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.BatchNorm1d(HIDDEN_DIM),

            nn.Linear(HIDDEN_DIM, OUTPUT_DIM)
        )

    def forward(self, x):
        return self.model(x)

# =========================
# 3. 데이터 준비
# =========================
# ⚠️ 여기 부분은 네가 실제 데이터로 교체하면 된다.

# 예시용 더미 데이터 (나중에 삭제)
num_samples = 1000
X = np.random.randn(num_samples, INPUT_DIM).astype(np.float32)
y = np.random.randint(0, OUTPUT_DIM, size=(num_samples,)).astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# =========================
# 4. 모델, 손실함수, 옵티마이저
# =========================
model = MLP().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# =========================
# 5. 학습 루프
# =========================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # =========================
    # 6. 테스트 정확도 계산
    # =========================
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Loss: {avg_loss:.4f} "
          f"Test Accuracy: {accuracy:.2f}%")

print("Training Finished.")


# 연습1 : 평균만 고려한 상태에서 구간을 맞추는 mlp기반 신경망
- 이렇게 해봤을때는 랜덤한 상황으로 계속 noise가 나오고 이것을 잡을 만큼의 입력 특징이 부족한것일수있고 즉 이 상황을 학습할만한 수렴적인 손실함수가 나오지 않아 지속적으로 학습해도 계속 학습 붕괴가 일어나는것같다.

In [168]:
import random
import numpy as np

# -------------------------
# 설정
# -------------------------
p = 1/125
q = 1 - p

intervals = [(1, 28), (29, 64), (65, 115), (116, 200), (201, float('inf'))]

NUM_GAMES = 100000   # 게임 수 (원하는 만큼 늘려도 됨)

# -------------------------
# 유틸 함수
# -------------------------

def make_jackpot_num():
    """기하분포(p=1/125)를 시뮬레이션해서 첫 잭팟 시점을 반환"""
    jackpot_num = 1
    while True:
        if random.randint(1, 125) == 1:
            return jackpot_num
        jackpot_num += 1

def classify_interval(x):
    for i, (lo, hi) in enumerate(intervals):
        if lo <= x <= hi:
            return i
    return None

def moving_average(data, n):
    if len(data) == 0:
        return 0
    subset = data[-n:] if len(data) >= n else data
    return sum(subset) / len(subset)

# -------------------------
# 데이터 생성
# -------------------------

X_data = []
y_data = []

jackpot_history = []

for _ in range(NUM_GAMES):
    jackpot_num = make_jackpot_num()
    jackpot_history.append(jackpot_num)

    # 최근 평균 계산
    avg5  = moving_average(jackpot_history, 5)
    avg10 = moving_average(jackpot_history, 10)
    avg15 = moving_average(jackpot_history, 15)
    avg20 = moving_average(jackpot_history, 20)

    # 125 중심화
    features = [
        avg5 - 125,
        avg10 - 125,
        avg15 - 125,
        avg20 - 125
    ]

    X_data.append(features)

    # 정답: 이번 jackpot_num이 속하는 구간
    label = classify_interval(jackpot_num)
    y_data.append(label)

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int64)

print("데이터 생성 완료")
print("X shape:", X_data.shape)
print("y shape:", y_data.shape)


데이터 생성 완료
X shape: (100000, 4)
y shape: (100000,)


In [169]:
for i in range(30):
    print(f"X[{i}] = {X_data[i]},  y[{i}] = {y_data[i]}")


X[0] = [24. 24. 24. 24.],  y[0] = 3
X[1] = [83. 83. 83. 83.],  y[1] = 4
X[2] = [31. 31. 31. 31.],  y[2] = 1
X[3] = [-5. -5. -5. -5.],  y[3] = 0
X[4] = [-23.6 -23.6 -23.6 -23.6],  y[4] = 0
X[5] = [-52.2 -39.5 -39.5 -39.5],  y[5] = 0
X[6] = [-83.4      -35.857143 -35.857143 -35.857143],  y[6] = 2
X[7] = [-67.   -30.25 -30.25 -30.25],  y[7] = 3
X[8] = [-46.       -27.777779 -27.777779 -27.777779],  y[8] = 3
X[9] = [-10.2 -16.9 -16.9 -16.9],  y[9] = 4
X[10] = [68.        7.9       9.363636  9.363636],  y[10] = 4
X[11] = [87.4  2.  15.5 15.5],  y[11] = 4
X[12] = [63.8      -1.6       5.923077  5.923077],  y[12] = 0
X[13] = [48.8         1.4        -0.42857143 -0.42857143],  y[13] = 1
X[14] = [ 7.8      -1.2      -8.666667 -8.666667],  y[14] = 0
X[15] = [-66.2    0.9  -16.8  -14.25],  y[15] = 0
X[16] = [-94.4       -3.5      -30.133333 -16.82353 ],  y[16] = 2
X[17] = [-77.6       -6.9      -26.933332 -17.277779],  y[17] = 2
X[18] = [-73.8      -12.5      -23.666666 -19.736841],  y[18] = 1
X[

In [170]:
# ✅ Step 1 — train/test 시간 분리

from sklearn.model_selection import train_test_split

# 시간 순서 유지
split_ratio = 0.8
split_idx = int(len(X_data) * split_ratio)

X_train = X_data[:split_idx]
X_test  = X_data[split_idx:]

y_train = y_data[:split_idx]
y_test  = y_data[split_idx:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (80000, 4)
Test size: (20000, 4)


In [171]:
# ✅ Step 2 — 표준화

# train 기준 평균, 표준편차 계산
mean = X_train.mean(axis=0)
std  = X_train.std(axis=0)

# 0으로 나누는 것 방지
std[std == 0] = 1e-8

# 표준화
X_train_std = (X_train - mean) / std
X_test_std  = (X_test  - mean) / std

print("표준화 완료")
print("Train mean:", X_train_std.mean(axis=0))
print("Train std:", X_train_std.std(axis=0))

for i in range(30):
    print("Before:", X_train[i])
    print("After :", X_train_std[i])
    print("Label :", y_train[i])
    print("------")


표준화 완료
Train mean: [-9.188056e-09  4.844461e-09  9.154901e-09  4.242435e-08]
Train std: [0.9999928 0.9999957 1.0000124 1.0000063]
Before: [24. 24. 24. 24.]
After : [0.43503597 0.6146641  0.7519514  0.86632466]
Label : 3
------
Before: [83. 83. 83. 83.]
After : [1.4985037 2.1171787 2.5900226 2.9839053]
Label : 4
------
Before: [31. 31. 31. 31.]
After : [0.5612101 0.7929285 0.9700277 1.117563 ]
Label : 1
------
Before: [-5. -5. -5. -5.]
After : [-0.08768546 -0.12386014 -0.1515073  -0.17452005]
Label : 0
------
Before: [-23.6 -23.6 -23.6 -23.6]
After : [-0.42294815 -0.5975343  -0.73096704 -0.84209627]
Label : 0
------
Before: [-52.2 -39.5 -39.5 -39.5]
After : [-0.9384597 -1.0024493 -1.2263117 -1.4127663]
Label : 0
------
Before: [-83.4      -35.857143 -35.857143 -35.857143]
After : [-1.5008359 -0.909679  -1.112823  -1.2820199]
Label : 2
------
Before: [-67.   -30.25 -30.25 -30.25]
After : [-1.2052279  -0.7668855  -0.93813944 -1.0807728 ]
Label : 3
------
Before: [-46.       -27.777779 -27

## 신경망 구조

In [142]:
# ✅ 1️⃣ PyTorch MLP 모델

import torch
import torch.nn as nn
import torch.optim as optim

# numpy → torch tensor
X_train_t = torch.tensor(X_train_std, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(X_test_std, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# -------------------------
# MLP 모델
# -------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )

    def forward(self, x):
        return self.net(x)

model = MLP()


In [172]:
# ✅ 1️⃣ PyTorch MLP 모델 (GELU)

import torch
import torch.nn as nn
import torch.optim as optim

# numpy → torch tensor
X_train_t = torch.tensor(X_train_std, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(X_test_std, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# -------------------------
# MLP 모델 (GELU)
# -------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128),
            nn.GELU(),          # 🔥 ReLU → GELU
            nn.Linear(128, 128),
            nn.GELU(),          # 🔥 ReLU → GELU
            nn.Linear(128, 5)
        )

    def forward(self, x):
        return self.net(x)

model = MLP()


## 손실함수와 옵티마이저

In [173]:
# ✅ 2️⃣ 손실함수 & 옵티마이저

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [174]:
# ✅ 3️⃣ 학습 루프

EPOCHS = 100
batch_size = 256

for epoch in range(EPOCHS):
    model.train()
    
    for i in range(0, len(X_train_t), batch_size):
        xb = X_train_t[i:i+batch_size]
        yb = y_train_t[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    # 매 epoch마다 test accuracy 계산
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t)
        _, predicted = torch.max(outputs, 1)
        correct = (predicted == y_test_t).sum().item()
        acc = correct / len(y_test_t)

    print(f"Epoch {epoch+1}/{EPOCHS}, Test Accuracy: {acc:.4f}")


Epoch 1/100, Test Accuracy: 0.3029
Epoch 2/100, Test Accuracy: 0.3039
Epoch 3/100, Test Accuracy: 0.3056
Epoch 4/100, Test Accuracy: 0.3066
Epoch 5/100, Test Accuracy: 0.3060
Epoch 6/100, Test Accuracy: 0.3063
Epoch 7/100, Test Accuracy: 0.3063
Epoch 8/100, Test Accuracy: 0.3067
Epoch 9/100, Test Accuracy: 0.3066
Epoch 10/100, Test Accuracy: 0.3069
Epoch 11/100, Test Accuracy: 0.3068
Epoch 12/100, Test Accuracy: 0.3068
Epoch 13/100, Test Accuracy: 0.3066
Epoch 14/100, Test Accuracy: 0.3070
Epoch 15/100, Test Accuracy: 0.3066
Epoch 16/100, Test Accuracy: 0.3069
Epoch 17/100, Test Accuracy: 0.3065
Epoch 18/100, Test Accuracy: 0.3070
Epoch 19/100, Test Accuracy: 0.3065
Epoch 20/100, Test Accuracy: 0.3064
Epoch 21/100, Test Accuracy: 0.3059
Epoch 22/100, Test Accuracy: 0.3058
Epoch 23/100, Test Accuracy: 0.3058
Epoch 24/100, Test Accuracy: 0.3061
Epoch 25/100, Test Accuracy: 0.3069
Epoch 26/100, Test Accuracy: 0.3068
Epoch 27/100, Test Accuracy: 0.3065
Epoch 28/100, Test Accuracy: 0.3064
E

In [175]:
# 🎯 4️⃣ Baseline Accuracy도 계산하자

import numpy as np

# 가장 많이 등장하는 클래스
unique, counts = np.unique(y_train, return_counts=True)
major_class = unique[np.argmax(counts)]

baseline_acc = np.mean(y_test == major_class)
print("Baseline accuracy:", baseline_acc)


Baseline accuracy: 0.20245


In [176]:
model.eval()
with torch.no_grad():
    outputs = model(X_test_t)
    _, predicted = torch.max(outputs, 1)
    acc = (predicted == y_test_t).float().mean()
print("Model Test Accuracy:", acc.item())


Model Test Accuracy: 0.30344998836517334


In [177]:
model.eval()

MLP(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=128, out_features=5, bias=True)
  )
)

# 새롭게 만든 젝팟넘에 위에서 학습시킨 가중치 가져와서 실험해보기
- 연습1 신경망 모델은 새롭게 만든 데이터에 대해서는 30%정확도에서 다시 정확도가 20%가까히로 떨어짐

In [178]:
def generate_new_dataset(num_samples=50000):

    X_new = []
    y_new = []
    jackpot_history = []

    for _ in range(num_samples):

        k = make_jackpot_num()
        jackpot_history.append(k)

        if len(jackpot_history) >= 20:
            ma5  = moving_average(jackpot_history[:-1], 5)
            ma10 = moving_average(jackpot_history[:-1], 10)
            ma15 = moving_average(jackpot_history[:-1], 15)
            ma20 = moving_average(jackpot_history[:-1], 20)

            # 🔥 125 중심화
            features = [
                ma5  - 125,
                ma10 - 125,
                ma15 - 125,
                ma20 - 125
            ]

            X_new.append(features)
            y_new.append(classify_interval(k))

    return np.array(X_new, dtype=np.float32), np.array(y_new, dtype=np.int64)


In [179]:
# 1️⃣ 새 데이터 생성
X_new, y_new = generate_new_dataset(50000)

# 2️⃣ 생성 확인
print("총 샘플 수:", len(X_new))
print("")

for i in range(10):
    interval_idx = y_new[i]
    lo, hi = intervals[interval_idx]

    print(f"Sample {i+1}")
    print("Input (ma5-125, ma10-125, ma15-125, ma20-125):", X_new[i])
    print(f"Label (interval index): {interval_idx} ({lo} ~ {hi})")
    print("------")


총 샘플 수: 49981

Sample 1
Input (ma5-125, ma10-125, ma15-125, ma20-125): [  2.4       -10.4        17.866667    3.4736843]
Label (interval index): 0 (1 ~ 28)
------
Sample 2
Input (ma5-125, ma10-125, ma15-125, ma20-125): [-12.6       -29.9         0.6666667  -2.85     ]
Label (interval index): 0 (1 ~ 28)
------
Sample 3
Input (ma5-125, ma10-125, ma15-125, ma20-125): [-43.4      -37.3      -22.733334  -5.3     ]
Label (interval index): 1 (29 ~ 64)
------
Sample 4
Input (ma5-125, ma10-125, ma15-125, ma20-125): [-84.2      -34.1      -28.866667 -10.2     ]
Label (interval index): 1 (29 ~ 64)
------
Sample 5
Input (ma5-125, ma10-125, ma15-125, ma20-125): [-89.6      -48.2      -29.666666 -11.5     ]
Label (interval index): 2 (65 ~ 115)
------
Sample 6
Input (ma5-125, ma10-125, ma15-125, ma20-125): [-90.2  -43.9  -37.    -9.15]
Label (interval index): 4 (201 ~ inf)
------
Sample 7
Input (ma5-125, ma10-125, ma15-125, ma20-125): [-14.2      -13.4      -24.666666  -3.05    ]
Label (interval inde

In [180]:
print("\n🔍 사람이 이해하기 쉽게 출력\n")

for i in range(10):
    interval_idx = y_new[i]
    lo, hi = intervals[interval_idx]

    print(f"Sample {i+1}")
    print("평균 features:", X_new[i])
    print(f"정답 구간: {interval_idx} ({lo} ~ {hi})")
    print("------")



🔍 사람이 이해하기 쉽게 출력

Sample 1
평균 features: [  2.4       -10.4        17.866667    3.4736843]
정답 구간: 0 (1 ~ 28)
------
Sample 2
평균 features: [-12.6       -29.9         0.6666667  -2.85     ]
정답 구간: 0 (1 ~ 28)
------
Sample 3
평균 features: [-43.4      -37.3      -22.733334  -5.3     ]
정답 구간: 1 (29 ~ 64)
------
Sample 4
평균 features: [-84.2      -34.1      -28.866667 -10.2     ]
정답 구간: 1 (29 ~ 64)
------
Sample 5
평균 features: [-89.6      -48.2      -29.666666 -11.5     ]
정답 구간: 2 (65 ~ 115)
------
Sample 6
평균 features: [-90.2  -43.9  -37.    -9.15]
정답 구간: 4 (201 ~ inf)
------
Sample 7
평균 features: [-14.2      -13.4      -24.666666  -3.05    ]
정답 구간: 4 (201 ~ inf)
------
Sample 8
평균 features: [ 28.2       -7.6      -15.466666 -10.      ]
정답 구간: 0 (1 ~ 28)
------
Sample 9
평균 features: [ 17.8 -33.2 -16.8 -17.2]
정답 구간: 2 (65 ~ 115)
------
Sample 10
평균 features: [ 31.8      -28.9      -21.533333 -14.3     ]
정답 구간: 0 (1 ~ 28)
------


In [181]:
# ✅ 4️⃣ 기존 표준화 파라미터 사용

X_new_std = (X_new - mean) / std

In [165]:
X_new_t = torch.tensor(X_new_std, dtype=torch.float32)
y_new_t = torch.tensor(y_new, dtype=torch.long)


In [166]:
# ✅ 6️⃣ 예측 및 정확도 계산

with torch.no_grad():
    outputs = model(X_new_t)
    preds = torch.argmax(outputs, dim=1)

    acc = (preds == y_new_t).float().mean()

print("New Data Accuracy:", acc.item())


New Data Accuracy: 0.20099638402462006


## 위에서 학습된 가중치 출력해보기

In [167]:
print(model.net[0].weight[:5])

tensor([[-1.0364, -0.0984, -0.0317,  0.1507],
        [ 0.2993, -0.4680,  0.1138,  0.1137],
        [ 0.0246, -0.6159,  0.1384,  0.0272],
        [-0.8130, -0.0995, -0.0370, -0.0039],
        [ 0.1368, -0.1535,  0.6090,  0.3463]], grad_fn=<SliceBackward0>)


# 연습2 : 랜덤한 상황을 통제할수있는 구간 느낌의 입력값을 추가하고 신경망을 이용해 이것을 이용할수있도록 구성하여(역전파관점 순전파관점에서 각 신경망의 위치와 형태가 어떻게 이용될것인지를 이해하고 습득하기) 학습해보자
- 즉 랜덤인 상황을 학습하려하면 극단적인 상황으로 인해 noise가 커서 배치단위 학습이다보니 수렴적인 손실함수가 나오기 어려워 학습이 붕괴하는데 만약 입력값에 이런 noise를 통제할수있는 랜덤적인 상황에 규칙을 찾을수있게 해주는 입력이 들어가면 학습이 안정화 될수도 있겠어

- 구간분포를 학습 입력값으로 추가 하려는데 이것을 원래 확률로 매핑해서 학습시키려했으나 그렇게하면 같은 확률인경우 구간분포를 구분하기 어렵게되어 정보가 소실되는 현상이 생겨 원래 데이터 대로 구간분포를 점별곱이나 가중치를 각각 다르게 주어 점별곱하는 방식으로 학습에 유용하게 구성하여 사용해야겠다.

- 

In [1]:
import random
import numpy as np

# -------------------------
# 설정
# -------------------------
p = 1/125
q = 1 - p

intervals = [(1, 28), (29, 64), (65, 115), (116, 200), (201, float('inf'))]

NUM_GAMES = 100000

# -------------------------
# 유틸 함수
# -------------------------

def make_jackpot_num():
    jackpot_num = 1
    while True:
        if random.randint(1, 125) == 1:
            return jackpot_num
        jackpot_num += 1

def classify_interval(x):
    for i, (lo, hi) in enumerate(intervals):
        if lo <= x <= hi:
            return i
    return None

def moving_average(data, n):
    if len(data) == 0:
        return 0
    subset = data[-n:] if len(data) >= n else data
    return sum(subset) / len(subset)

def interval_distribution(data, n):
    """
    최근 n개 jackpot_num이 속한 구간 분포 (길이 5 벡터 반환)
    """
    recent = data[-n:]
    counts = [0] * len(intervals)

    for val in recent:
        idx = classify_interval(val)
        if idx is not None:
            counts[idx] += 1

    return counts

# -------------------------
# 데이터 생성
# -------------------------

X_data = []
y_data = []

jackpot_history = []

for _ in range(NUM_GAMES):
    jackpot_num = make_jackpot_num()
    jackpot_history.append(jackpot_num)

    # -------------------------
    # 🔥 1️⃣ 구간 분포 (20차원)
    # -------------------------
    dist5  = interval_distribution(jackpot_history, 5)
    dist10 = interval_distribution(jackpot_history, 10)
    dist15 = interval_distribution(jackpot_history, 15)
    dist20 = interval_distribution(jackpot_history, 20)

    interval_features = dist5 + dist10 + dist15 + dist20  # 총 20차원

    # -------------------------
    # 2️⃣ 평균 중심화 (4차원)
    # -------------------------
    avg5  = moving_average(jackpot_history, 5)
    avg10 = moving_average(jackpot_history, 10)
    avg15 = moving_average(jackpot_history, 15)
    avg20 = moving_average(jackpot_history, 20)

    avg_features = [
        avg5 - 125,
        avg10 - 125,
        avg15 - 125,
        avg20 - 125
    ]

    # -------------------------
    # ✅ 최종 입력 벡터 (24차원)
    # -------------------------
    features = interval_features + avg_features

    X_data.append(features)

    # 정답: 이번 jackpot_num 구간
    label = classify_interval(jackpot_num)
    y_data.append(label)

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int64)

print("데이터 생성 완료")
print("X shape:", X_data.shape)  # (NUM_GAMES, 24)
print("y shape:", y_data.shape)


데이터 생성 완료
X shape: (100000, 24)
y shape: (100000,)


In [2]:
# 한 칸 shift(기존 x[0] = y[0] 대응에서 x[0] = y[1]로 대응시키도록 바꾸기)
X_data = X_data[:-1]     # 마지막 X는 버림
y_data = y_data[1:]      # 첫 y는 버림

print("Shifted X shape:", X_data.shape)
print("Shifted y shape:", y_data.shape)

Shifted X shape: (99999, 24)
Shifted y shape: (99999,)


In [288]:
for i in range(30):
    print(f"X[{i}] = {X_data[i]},  y[{i}] = {y_data[i]}")


X[0] = [  0.   0.   0.   0.   1.   0.   0.   0.   0.   1.   0.   0.   0.   0.
   1.   0.   0.   0.   0.   1. 122. 122. 122. 122.],  y[0] = 1
X[1] = [ 0.   1.   0.   0.   1.   0.   1.   0.   0.   1.   0.   1.   0.   0.
  1.   0.   1.   0.   0.   1.  13.5 13.5 13.5 13.5],  y[1] = 2
X[2] = [ 0.  1.  1.  0.  1.  0.  1.  1.  0.  1.  0.  1.  1.  0.  1.  0.  1.  1.
  0.  1. -3. -3. -3. -3.],  y[2] = 0
X[3] = [  1.     1.     1.     0.     1.     1.     1.     1.     0.     1.
   1.     1.     1.     0.     1.     1.     1.     1.     0.     1.
 -28.75 -28.75 -28.75 -28.75],  y[3] = 0
X[4] = [  2.    1.    1.    0.    1.    2.    1.    1.    0.    1.    2.    1.
   1.    0.    1.    2.    1.    1.    0.    1.  -47.6 -47.6 -47.6 -47.6],  y[4] = 2
X[5] = [  2.   1.   2.   0.   0.   2.   1.   2.   0.   1.   2.   1.   2.   0.
   1.   2.   1.   2.   0.   1. -76. -43. -43. -43.],  y[5] = 3
X[6] = [  2.   0.   2.   1.   0.   2.   1.   2.   1.   1.   2.   1.   2.   1.
   1.   2.   1.   2.   1.   1. -4

In [289]:
# -------------------------
# Step 1 — train/test 시간 분리
# -------------------------

split_ratio = 0.8
split_idx = int(len(X_data) * split_ratio)

X_train = X_data[:split_idx]
X_test  = X_data[split_idx:]

# 🔥 정수 라벨 기준으로 분리 (원핫 제거)
y_train = y_data[:split_idx]
y_test  = y_data[split_idx:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


Train size: (79999, 24)
Test size: (20000, 24)
y_train shape: (79999,)
y_test shape: (20000,)


In [290]:
# -------------------------
# Step 2 — 부분 표준화 (마지막 4개만)
# -------------------------

# 마지막 4개 평균 피처에 대해서만 계산
mean_last4 = X_train[:, -4:].mean(axis=0)
std_last4  = X_train[:, -4:].std(axis=0)

std_last4[std_last4 == 0] = 1e-8

# 복사해서 수정
X_train_std = X_train.copy()
X_test_std  = X_test.copy()

# 마지막 4개만 표준화 적용
X_train_std[:, -4:] = (X_train[:, -4:] - mean_last4) / std_last4
X_test_std[:, -4:]  = (X_test[:, -4:]  - mean_last4) / std_last4

print("부분 표준화 완료")
print("Train last4 mean:", X_train_std[:, -4:].mean(axis=0))
print("Train last4 std:", X_train_std[:, -4:].std(axis=0))


부분 표준화 완료
Train last4 mean: [-3.8289013e-08 -4.8166742e-09  4.7874305e-08 -2.4855821e-08]
Train last4 std: [1.0000061  1.0000038  0.99999696 0.99998695]


In [291]:
# ✅ 1️⃣ PyTorch MLP 모델 (GELU)

import torch
import torch.nn as nn
import torch.optim as optim

# numpy → torch tensor
X_train_t = torch.tensor(X_train_std, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(X_test_std, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# -------------------------
# MLP 모델 (GELU)
# -------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(24, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(64, 32),
            nn.GELU(),

            nn.Linear(32, 5)
        )

    def forward(self, x):
        return self.net(x)

model = MLP()


In [292]:
'''
# -------------------------
# 클래스 가중치 계산 (기하분포라서 각 구간(클래스) 분포가 달라 즉 정답개수가 앞쪽에 몰려있을수 있어 고른 학습을 위해 정답이 4,5와같이 뒤에있는것들이 학습될때 그 학습률을 늘리는 기법

# -------------------------
class_counts = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()

class_weights = torch.tensor(class_weights, dtype=torch.float32)
'''

# -------------------------
# 손실함수 (위에 가중학습 기능 제거)
# -------------------------
criterion = nn.CrossEntropyLoss()

# -------------------------
# 옵티마이저
# -------------------------
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

In [280]:
'''
EPOCHS = 40
batch_size = 256

for epoch in range(EPOCHS):
    model.train()
    
    epoch_loss = 0
    correct_train = 0

    # 🔵 순서 그대로 사용 (shuffle 제거) => SGD안하는것임 시계열학습 염두해두고있기때문에 학습데이터 안섞는것
    for i in range(0, len(X_train_t), batch_size):
        xb = X_train_t[i:i+batch_size]
        yb = y_train_t[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == yb).sum().item()

    train_acc = correct_train / len(X_train_t)
'''


'\nEPOCHS = 40\nbatch_size = 256\n\nfor epoch in range(EPOCHS):\n    model.train()\n    \n    epoch_loss = 0\n    correct_train = 0\n\n    # 🔵 순서 그대로 사용 (shuffle 제거) => SGD안하는것임 시계열학습 염두해두고있기때문에 학습데이터 안섞는것\n    for i in range(0, len(X_train_t), batch_size):\n        xb = X_train_t[i:i+batch_size]\n        yb = y_train_t[i:i+batch_size]\n\n        optimizer.zero_grad()\n        outputs = model(xb)\n        loss = criterion(outputs, yb)\n        loss.backward()\n        optimizer.step()\n\n        epoch_loss += loss.item()\n\n        _, predicted = torch.max(outputs, 1)\n        correct_train += (predicted == yb).sum().item()\n\n    train_acc = correct_train / len(X_train_t)\n'

In [293]:
EPOCHS = 40
batch_size = 256

for epoch in range(EPOCHS):
    print("Loop running:", epoch)

    model.train()
    epoch_loss = 0
    correct_train = 0

    for i in range(0, len(X_train_t), batch_size):
        xb = X_train_t[i:i+batch_size]
        yb = y_train_t[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        _, predicted_train = torch.max(outputs, 1)
        correct_train += (predicted_train == yb).sum().item()

    train_acc = correct_train / len(X_train_t)

    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t)
        _, predicted_test = torch.max(outputs, 1)
        correct_test = (predicted_test == y_test_t).sum().item()
        test_acc = correct_test / len(y_test_t)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {epoch_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f}")


Loop running: 0
Epoch 1/40 | Loss: 503.8716 | Train Acc: 0.2091 | Test Acc: 0.2024
Loop running: 1
Epoch 2/40 | Loss: 503.0635 | Train Acc: 0.2156 | Test Acc: 0.2051
Loop running: 2
Epoch 3/40 | Loss: 502.6864 | Train Acc: 0.2186 | Test Acc: 0.2041
Loop running: 3
Epoch 4/40 | Loss: 502.3687 | Train Acc: 0.2210 | Test Acc: 0.2007
Loop running: 4
Epoch 5/40 | Loss: 502.1454 | Train Acc: 0.2247 | Test Acc: 0.2004
Loop running: 5
Epoch 6/40 | Loss: 501.9305 | Train Acc: 0.2253 | Test Acc: 0.2003
Loop running: 6
Epoch 7/40 | Loss: 501.5167 | Train Acc: 0.2282 | Test Acc: 0.2026
Loop running: 7
Epoch 8/40 | Loss: 501.3672 | Train Acc: 0.2327 | Test Acc: 0.2024
Loop running: 8
Epoch 9/40 | Loss: 501.1590 | Train Acc: 0.2316 | Test Acc: 0.2015
Loop running: 9
Epoch 10/40 | Loss: 500.9710 | Train Acc: 0.2315 | Test Acc: 0.1996
Loop running: 10
Epoch 11/40 | Loss: 500.6430 | Train Acc: 0.2346 | Test Acc: 0.2003
Loop running: 11
Epoch 12/40 | Loss: 500.3137 | Train Acc: 0.2358 | Test Acc: 0.2031

In [294]:
# 🎯 4️⃣ Baseline Accuracy도 계산하자

import numpy as np

# 가장 많이 등장하는 클래스
unique, counts = np.unique(y_train, return_counts=True)
major_class = unique[np.argmax(counts)]

baseline_acc = np.mean(y_test == major_class)
print("Baseline accuracy:", baseline_acc)

print("Major class:", major_class)

Baseline accuracy: 0.2051
Major class: 0


# 연습2 성능 확인

In [295]:
# -------------------------
# 🔎 Test 상세 분석
# -------------------------

model.eval()
with torch.no_grad():
    outputs = model(X_test_t)
    probs = torch.softmax(outputs, dim=1)  # 각 클래스 확률
    _, predicted = torch.max(outputs, 1)

# 맞았는지 여부
correct_mask = (predicted == y_test_t)

print("\n===== Test Detailed Results (앞 50개) =====\n")

for i in range(50):
    print(f"Index {i}")
    print(f"  True Label      : {y_test_t[i].item()}")
    print(f"  Predicted Label : {predicted[i].item()}")
    print(f"  Correct         : {correct_mask[i].item()}")
    print(f"  Probabilities   : {probs[i].numpy()}")
    print("-" * 40)



===== Test Detailed Results (앞 50개) =====

Index 0
  True Label      : 2
  Predicted Label : 0
  Correct         : False
  Probabilities   : [0.23278967 0.22324568 0.19484612 0.17801458 0.17110398]
----------------------------------------
Index 1
  True Label      : 2
  Predicted Label : 1
  Correct         : False
  Probabilities   : [0.1930249  0.23493919 0.1984212  0.1914729  0.18214186]
----------------------------------------
Index 2
  True Label      : 0
  Predicted Label : 1
  Correct         : False
  Probabilities   : [0.17749873 0.25962383 0.20112884 0.17555687 0.18619172]
----------------------------------------
Index 3
  True Label      : 4
  Predicted Label : 1
  Correct         : False
  Probabilities   : [0.17259164 0.2456467  0.18810667 0.1951316  0.19852337]
----------------------------------------
Index 4
  True Label      : 3
  Predicted Label : 3
  Correct         : True
  Probabilities   : [0.1637749  0.18742265 0.2209543  0.24619904 0.18164915]
------------------

In [296]:
# 2. confusion matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_t.numpy(), predicted.numpy())

print("\nConfusion Matrix:\n")
print(cm)

'''
행 = True class
열 = Predicted class
행렬 결과 숫자들 모두 합하면 20000나옴 위 test 샘플 숫자임 test sample 어디서 맞고 틀렸나 분포 알수있는것임
이 결과보면 모든 클래스(구간)에서 균등한 정확도를 보이는데 딱히 적은 클래스 데이터 학습시 가중치주는 기법 안넣어도 잘 학습됨 GPT가 꼭 맞는것은 아님 그져 추천하는 애일뿐 다 예상하고 맞추지도 못한다는것을 이번 프로젝트를 통해 알수있었음
'''


Confusion Matrix:

[[ 896 1035  775  755  641]
 [ 898  973  732  728  601]
 [ 903  987  755  789  625]
 [ 909  950  766  728  580]
 [ 907  959  702  769  637]]


'\n행 = True class\n열 = Predicted class\n행렬 결과 숫자들 모두 합하면 20000나옴 위 test 샘플 숫자임 test sample 어디서 맞고 틀렸나 분포 알수있는것임\n이 결과보면 모든 클래스(구간)에서 균등한 정확도를 보이는데 딱히 적은 클래스 데이터 학습시 가중치주는 기법 안넣어도 잘 학습됨 GPT가 꼭 맞는것은 아님 그져 추천하는 애일뿐 다 예상하고 맞추지도 못한다는것을 이번 프로젝트를 통해 알수있었음\n'

In [297]:
# 3. 틀린 샘플만 따로보기

print("\n===== 틀린 샘플 일부 =====\n")

wrong_indices = (predicted != y_test_t).nonzero(as_tuple=True)[0]

for i in wrong_indices[:20]:
    print(f"Index {i.item()}")
    print(f"  True      : {y_test_t[i].item()}")
    print(f"  Predicted : {predicted[i].item()}")
    print(f"  Prob      : {probs[i].numpy()}")
    print("-" * 40)



===== 틀린 샘플 일부 =====

Index 0
  True      : 2
  Predicted : 0
  Prob      : [0.23278967 0.22324568 0.19484612 0.17801458 0.17110398]
----------------------------------------
Index 1
  True      : 2
  Predicted : 1
  Prob      : [0.1930249  0.23493919 0.1984212  0.1914729  0.18214186]
----------------------------------------
Index 2
  True      : 0
  Predicted : 1
  Prob      : [0.17749873 0.25962383 0.20112884 0.17555687 0.18619172]
----------------------------------------
Index 3
  True      : 4
  Predicted : 1
  Prob      : [0.17259164 0.2456467  0.18810667 0.1951316  0.19852337]
----------------------------------------
Index 5
  True      : 2
  Predicted : 0
  Prob      : [0.21419187 0.18332483 0.19395937 0.20351794 0.20500596]
----------------------------------------
Index 6
  True      : 0
  Predicted : 4
  Prob      : [0.20102416 0.17596442 0.2048246  0.18762265 0.23056418]
----------------------------------------
Index 7
  True      : 1
  Predicted : 0
  Prob      : [0.24076657

## 이제 두번째 모델로도 새로운 데이터 만들어서 일반화 성공했는지 시험해봐야함 안됐으면 이번에는 신경망구조 손보고 연속정도도 지표로 추가해서 학습시켜보자

# 연습 2에서 마지막 네개 입력제거해도 성능 같은지 확인해보기 => 결과확인해보니 정확도 거의같음

In [298]:
import random
import numpy as np

# -------------------------
# 설정
# -------------------------
p = 1/125
q = 1 - p

intervals = [(1, 28), (29, 64), (65, 115), (116, 200), (201, float('inf'))]

NUM_GAMES = 100000

# -------------------------
# 유틸 함수
# -------------------------

def make_jackpot_num():
    jackpot_num = 1
    while True:
        if random.randint(1, 125) == 1:
            return jackpot_num
        jackpot_num += 1

def classify_interval(x):
    for i, (lo, hi) in enumerate(intervals):
        if lo <= x <= hi:
            return i
    return None

def interval_distribution(data, n):
    recent = data[-n:]
    counts = [0] * len(intervals)

    for val in recent:
        idx = classify_interval(val)
        if idx is not None:
            counts[idx] += 1

    return counts

# -------------------------
# 데이터 생성 (20차원 입력)
# -------------------------

X_data = []
y_data = []

jackpot_history = []

for _ in range(NUM_GAMES):

    jackpot_num = make_jackpot_num()
    jackpot_history.append(jackpot_num)

    # 🔥 구간 분포 20차원만 사용
    dist5  = interval_distribution(jackpot_history, 5)
    dist10 = interval_distribution(jackpot_history, 10)
    dist15 = interval_distribution(jackpot_history, 15)
    dist20 = interval_distribution(jackpot_history, 20)

    interval_features = dist5 + dist10 + dist15 + dist20  # 총 20차원

    # 평균 feature 제거
    features = interval_features

    X_data.append(features)

    label = classify_interval(jackpot_num)
    y_data.append(label)

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int64)

print("데이터 생성 완료 (20차원)")
print("X shape:", X_data.shape)  # (NUM_GAMES, 20)
print("y shape:", y_data.shape)


데이터 생성 완료 (20차원)
X shape: (100000, 20)
y shape: (100000,)


In [299]:
# 한 칸 shift(기존 x[0] = y[0] 대응에서 x[0] = y[1]로 대응시키도록 바꾸기)
X_data = X_data[:-1]     # 마지막 X는 버림
y_data = y_data[1:]      # 첫 y는 버림

print("Shifted X shape:", X_data.shape)
print("Shifted y shape:", y_data.shape)

Shifted X shape: (99999, 20)
Shifted y shape: (99999,)


In [300]:
for i in range(30):
    print(f"X[{i}] = {X_data[i]},  y[{i}] = {y_data[i]}")


X[0] = [0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0.],  y[0] = 0
X[1] = [1. 0. 0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 1. 0.],  y[1] = 3
X[2] = [1. 0. 0. 2. 0. 1. 0. 0. 2. 0. 1. 0. 0. 2. 0. 1. 0. 0. 2. 0.],  y[2] = 2
X[3] = [1. 0. 1. 2. 0. 1. 0. 1. 2. 0. 1. 0. 1. 2. 0. 1. 0. 1. 2. 0.],  y[3] = 0
X[4] = [2. 0. 1. 2. 0. 2. 0. 1. 2. 0. 2. 0. 1. 2. 0. 2. 0. 1. 2. 0.],  y[4] = 4
X[5] = [2. 0. 1. 1. 1. 2. 0. 1. 2. 1. 2. 0. 1. 2. 1. 2. 0. 1. 2. 1.],  y[5] = 3
X[6] = [1. 0. 1. 2. 1. 2. 0. 1. 3. 1. 2. 0. 1. 3. 1. 2. 0. 1. 3. 1.],  y[6] = 4
X[7] = [1. 0. 1. 1. 2. 2. 0. 1. 3. 2. 2. 0. 1. 3. 2. 2. 0. 1. 3. 2.],  y[7] = 3
X[8] = [1. 0. 0. 2. 2. 2. 0. 1. 4. 2. 2. 0. 1. 4. 2. 2. 0. 1. 4. 2.],  y[8] = 1
X[9] = [0. 1. 0. 2. 2. 2. 1. 1. 4. 2. 2. 1. 1. 4. 2. 2. 1. 1. 4. 2.],  y[9] = 2
X[10] = [0. 1. 1. 2. 1. 2. 1. 2. 3. 2. 2. 1. 2. 4. 2. 2. 1. 2. 4. 2.],  y[10] = 4
X[11] = [0. 1. 1. 1. 2. 1. 1. 2. 3. 3. 2. 1. 2. 4. 3. 2. 1. 2. 4. 3.],  y[11] = 1
X[12] = [0. 2. 1. 1. 1. 1. 2. 2. 2. 

In [301]:
# -------------------------
# Step 1 — train/test 시간 분리
# -------------------------

split_ratio = 0.8
split_idx = int(len(X_data) * split_ratio)

X_train = X_data[:split_idx]
X_test  = X_data[split_idx:]

# 🔥 정수 라벨 기준으로 분리 (원핫 제거)
y_train = y_data[:split_idx]
y_test  = y_data[split_idx:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


Train size: (79999, 20)
Test size: (20000, 20)
y_train shape: (79999,)
y_test shape: (20000,)


In [302]:
import torch
import torch.nn as nn
import torch.optim as optim

# numpy → torch tensor
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# -------------------------
# MLP 모델 (20차원 입력)
# -------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 128),      # 🔥 24 → 20 변경
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(64, 32),
            nn.GELU(),

            nn.Linear(32, 5)
        )

    def forward(self, x):
        return self.net(x)

model = MLP()


In [303]:
# -------------------------
# 손실함수 (위에 가중학습 기능 제거)
# -------------------------
criterion = nn.CrossEntropyLoss()

# -------------------------
# 옵티마이저
# -------------------------
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

In [304]:
EPOCHS = 40
batch_size = 256

for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0
    correct_train = 0

    # -------------------------
    # Train
    # -------------------------
    for i in range(0, len(X_train_t), batch_size):
        xb = X_train_t[i:i+batch_size]
        yb = y_train_t[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        _, predicted_train = torch.max(outputs, 1)
        correct_train += (predicted_train == yb).sum().item()

    train_acc = correct_train / len(X_train_t)

    # -------------------------
    # Test
    # -------------------------
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t)
        _, predicted_test = torch.max(outputs, 1)
        correct_test = (predicted_test == y_test_t).sum().item()
        test_acc = correct_test / len(y_test_t)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Loss: {epoch_loss:.4f} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f}")


Epoch 01/40 | Loss: 504.2323 | Train Acc: 0.2062 | Test Acc: 0.1996
Epoch 02/40 | Loss: 503.2690 | Train Acc: 0.2157 | Test Acc: 0.1997
Epoch 03/40 | Loss: 502.9060 | Train Acc: 0.2168 | Test Acc: 0.1996
Epoch 04/40 | Loss: 502.5535 | Train Acc: 0.2215 | Test Acc: 0.1984
Epoch 05/40 | Loss: 502.4134 | Train Acc: 0.2223 | Test Acc: 0.1978
Epoch 06/40 | Loss: 502.0084 | Train Acc: 0.2254 | Test Acc: 0.1972
Epoch 07/40 | Loss: 501.8356 | Train Acc: 0.2278 | Test Acc: 0.1993
Epoch 08/40 | Loss: 501.6242 | Train Acc: 0.2282 | Test Acc: 0.1978
Epoch 09/40 | Loss: 501.2330 | Train Acc: 0.2317 | Test Acc: 0.1969
Epoch 10/40 | Loss: 501.1320 | Train Acc: 0.2327 | Test Acc: 0.1976
Epoch 11/40 | Loss: 500.8527 | Train Acc: 0.2340 | Test Acc: 0.1984
Epoch 12/40 | Loss: 500.6195 | Train Acc: 0.2373 | Test Acc: 0.1998
Epoch 13/40 | Loss: 500.1498 | Train Acc: 0.2397 | Test Acc: 0.1966
Epoch 14/40 | Loss: 499.9212 | Train Acc: 0.2397 | Test Acc: 0.1966
Epoch 15/40 | Loss: 499.6874 | Train Acc: 0.2396

# 새롭게 만든 젝팟넘에 위에서 학습시킨 가중치 가져와서 실험해보기(연습2로)

## 일단은 성능이 45% 정확도로 유지됐다 GPT는 뭔가 설계가 잘못됐다고 하는데 확인해봐야겠다
- 입력 x와 정답 y가 대응이 잘못되었었다 x[0] = y[0]으로 되어 입력안에 정답이 있어 학습후 정확도가 45%로 높게나왔던것이다. x[0] = y[1]로 대응시키니 정확도가 다시 약 20%로 내려갔다.

In [284]:
# -------------------------
# 🔥 새로운 독립 데이터 생성
# -------------------------

NUM_NEW = 20000

X_new = []
y_new = []

jackpot_history_new = []

for _ in range(NUM_NEW):

    jackpot_num = make_jackpot_num()
    jackpot_history_new.append(jackpot_num)

    # 동일한 feature 생성 방식
    dist5  = interval_distribution(jackpot_history_new, 5)
    dist10 = interval_distribution(jackpot_history_new, 10)
    dist15 = interval_distribution(jackpot_history_new, 15)
    dist20 = interval_distribution(jackpot_history_new, 20)

    interval_features = dist5 + dist10 + dist15 + dist20

    avg5  = moving_average(jackpot_history_new, 5)
    avg10 = moving_average(jackpot_history_new, 10)
    avg15 = moving_average(jackpot_history_new, 15)
    avg20 = moving_average(jackpot_history_new, 20)

    avg_features = [
        avg5 - 125,
        avg10 - 125,
        avg15 - 125,
        avg20 - 125
    ]

    features = interval_features + avg_features

    X_new.append(features)
    y_new.append(classify_interval(jackpot_num))

X_new = np.array(X_new, dtype=np.float32)
y_new = np.array(y_new, dtype=np.int64)

print("New data shape:", X_new.shape)


New data shape: (20000, 24)


In [285]:
X_new_std = X_new.copy()
X_new_std[:, -4:] = (X_new[:, -4:] - mean_last4) / std_last4

In [286]:
X_new_t = torch.tensor(X_new_std, dtype=torch.float32)
y_new_t = torch.tensor(y_new, dtype=torch.long)

model.eval()
with torch.no_grad():
    outputs = model(X_new_t)
    _, predicted_new = torch.max(outputs, 1)
    correct_new = (predicted_new == y_new_t).sum().item()
    new_acc = correct_new / len(y_new_t)

print("🔥 New Independent Data Accuracy:", new_acc)


🔥 New Independent Data Accuracy: 0.45375


In [283]:
# 위에서 학습된 가중치 출력해보기

print(model.net[0].weight[:5])

tensor([[-3.4959e-01,  2.7140e-01, -5.9629e-02,  1.7237e-01, -8.1739e-02,
          7.3487e-02, -1.2107e-02,  1.7482e-01, -1.5051e-01, -8.9313e-02,
          1.4248e-02, -3.9513e-02,  1.4739e-02,  1.5185e-01,  8.4267e-02,
         -7.3984e-02, -9.3527e-02, -6.3650e-02, -1.4458e-01, -1.7674e-02,
          2.1019e-01, -1.0308e-01, -7.6468e-02,  3.1612e-02],
        [ 2.3058e-02, -1.9185e-01,  2.1003e-01,  2.1517e-01, -3.5623e-01,
          5.0835e-03, -1.5755e-01, -1.1815e-01, -1.9346e-01, -1.5178e-01,
          2.4283e-02,  2.0607e-02, -2.4973e-02, -1.0855e-02, -8.3531e-02,
         -9.2531e-02,  9.5441e-02, -2.5930e-02,  2.0425e-03,  1.0147e-01,
         -4.0567e-02,  2.3538e-01, -1.9201e-02, -9.4016e-02],
        [-1.5395e-01, -1.3352e-01,  1.5385e-01,  2.8357e-01, -2.4235e-01,
          2.1079e-01,  7.7088e-02,  1.2596e-02,  8.5337e-02, -9.6723e-02,
         -1.4561e-01,  9.4919e-02, -9.2876e-02, -1.3503e-01, -4.1372e-05,
         -9.2486e-02, -1.3664e-01,  7.4491e-02, -4.3609e-02,  

# 연습3 의도적인 신경망 설계해 학습시켜보기

In [1]:
import random
import numpy as np

# -------------------------
# 설정
# -------------------------
p = 1/125
q = 1 - p

intervals = [(1, 28), (29, 64), (65, 115), (116, 200), (201, float('inf'))]

NUM_GAMES = 100000

# -------------------------
# 유틸 함수
# -------------------------

def make_jackpot_num():
    jackpot_num = 1
    while True:
        if random.randint(1, 125) == 1:
            return jackpot_num
        jackpot_num += 1

def classify_interval(x):
    for i, (lo, hi) in enumerate(intervals):
        if lo <= x <= hi:
            return i
    return None

def interval_distribution(data, n):
    recent = data[-n:]
    counts = [0] * len(intervals)

    for val in recent:
        idx = classify_interval(val)
        if idx is not None:
            counts[idx] += 1

    return counts

# -------------------------
# 데이터 생성 (20차원 입력)
# -------------------------

X_data = []
y_data = []

jackpot_history = []

for _ in range(NUM_GAMES):

    jackpot_num = make_jackpot_num()
    jackpot_history.append(jackpot_num)

    # 🔥 구간 분포 20차원만 사용
    dist5  = interval_distribution(jackpot_history, 5)
    dist10 = interval_distribution(jackpot_history, 10)
    dist15 = interval_distribution(jackpot_history, 15)
    dist20 = interval_distribution(jackpot_history, 20)

    interval_features = dist5 + dist10 + dist15 + dist20  # 총 20차원

    # 평균 feature 제거
    features = interval_features

    X_data.append(features)

    label = classify_interval(jackpot_num)
    y_data.append(label)

X_data = np.array(X_data, dtype=np.float32)
y_data = np.array(y_data, dtype=np.int64)

print("데이터 생성 완료 (20차원)")
print("X shape:", X_data.shape)  # (NUM_GAMES, 20)
print("y shape:", y_data.shape)


데이터 생성 완료 (20차원)
X shape: (100000, 20)
y shape: (100000,)


In [8]:
# 한 칸 shift(기존 x[0] = y[0] 대응에서 x[0] = y[1]로 대응시키도록 바꾸기)
X_data = X_data[:-1]     # 마지막 X는 버림
y_data = y_data[1:]      # 첫 y는 버림

print("Shifted X shape:", X_data.shape)
print("Shifted y shape:", y_data.shape)

Shifted X shape: (99999, 20)
Shifted y shape: (99999,)


In [9]:
for i in range(30):
    print(f"X[{i}] = {X_data[i]},  y[{i}] = {y_data[i]}")


X[0] = [0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0.],  y[0] = 3
X[1] = [0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 1. 0. 1. 0.],  y[1] = 0
X[2] = [1. 1. 0. 1. 0. 1. 1. 0. 1. 0. 1. 1. 0. 1. 0. 1. 1. 0. 1. 0.],  y[2] = 4
X[3] = [1. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 1. 0. 1. 1.],  y[3] = 3
X[4] = [1. 1. 0. 2. 1. 1. 1. 0. 2. 1. 1. 1. 0. 2. 1. 1. 1. 0. 2. 1.],  y[4] = 2
X[5] = [1. 0. 1. 2. 1. 1. 1. 1. 2. 1. 1. 1. 1. 2. 1. 1. 1. 1. 2. 1.],  y[5] = 4
X[6] = [1. 0. 1. 1. 2. 1. 1. 1. 2. 2. 1. 1. 1. 2. 2. 1. 1. 1. 2. 2.],  y[6] = 4
X[7] = [0. 0. 1. 1. 3. 1. 1. 1. 2. 3. 1. 1. 1. 2. 3. 1. 1. 1. 2. 3.],  y[7] = 0
X[8] = [1. 0. 1. 1. 2. 2. 1. 1. 2. 3. 2. 1. 1. 2. 3. 2. 1. 1. 2. 3.],  y[8] = 3
X[9] = [1. 0. 1. 1. 2. 2. 1. 1. 3. 3. 2. 1. 1. 3. 3. 2. 1. 1. 3. 3.],  y[9] = 4
X[10] = [1. 0. 0. 1. 3. 2. 0. 1. 3. 4. 2. 1. 1. 3. 4. 2. 1. 1. 3. 4.],  y[10] = 3
X[11] = [1. 0. 0. 2. 2. 2. 0. 1. 3. 4. 2. 1. 1. 4. 4. 2. 1. 1. 4. 4.],  y[11] = 3
X[12] = [1. 0. 0. 3. 1. 1. 0. 1. 4. 

In [10]:
# -------------------------
# Step 1 — train/test 시간 분리
# -------------------------

split_ratio = 0.8
split_idx = int(len(X_data) * split_ratio)

X_train = X_data[:split_idx]
X_test  = X_data[split_idx:]

# 🔥 정수 라벨 기준으로 분리 (원핫 제거)
y_train = y_data[:split_idx]
y_test  = y_data[split_idx:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


Train size: (79999, 20)
Test size: (20000, 20)
y_train shape: (79999,)
y_test shape: (20000,)


In [11]:
import torch
import torch.nn as nn

class StructuredNet(nn.Module):
    def __init__(self):
        super().__init__()

        # 🔵 각 블록: 5 → 5 (구간공간 유지)
        self.block_proj = nn.ModuleList([
            nn.Linear(5, 5, bias=False) for _ in range(4)
        ])

        # 🔵 5차원 → MLP → 5차원
        self.mlp = nn.Sequential(
            nn.Linear(5, 32),
            nn.GELU(),
            
            nn.Linear(32, 16),
            nn.GELU(),
            
            nn.Linear(16, 5)
        )

    def forward(self, x):
        # x shape: (batch, 20)

        # 5차원씩 분리
        blocks = torch.chunk(x, 4, dim=1)  # 4개 (batch,5)

        # 각 블록을 5차원으로 선형변환 후 합산
        z = 0
        for i in range(4):
            z = z + self.block_proj[i](blocks[i])  # (batch,5)

        # MLP 통과
        out = self.mlp(z)

        return out


model = StructuredNet()

In [12]:
# -------------------------
# 손실함수
# -------------------------
criterion = nn.CrossEntropyLoss()

# -------------------------
# 옵티마이저
# -------------------------
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,          # 🔥 더 낮춤 (0.0005 → 0.0001)
    weight_decay=5e-3 # 🔥 강한 L2 (1e-4 → 5e-3)
)

In [20]:
EPOCHS = 200
batch_size = 1024

patience = 100                  # 🔥 몇 번까지 기다릴지
best_test_loss = float('inf')
early_stop_counter = 0

for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0
    correct_train = 0

    # -------------------------
    # Train
    # -------------------------
    for i in range(0, len(X_train_t), batch_size):   # 🔥 X_train → X_train_t
        xb = X_train_t[i:i+batch_size]               # 🔥 tensor 사용
        yb = y_train_t[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        _, predicted_train = torch.max(outputs, 1)
        correct_train += (predicted_train == yb).sum().item()

    train_acc = correct_train / len(X_train_t)       # 🔥 일관성 유지

    # -------------------------
    # Test
    # -------------------------
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t)                    # 🔥 tensor 사용
        test_loss = criterion(outputs, y_test_t)
        _, predicted_test = torch.max(outputs, 1)
        correct_test = (predicted_test == y_test_t).sum().item()
        test_acc = correct_test / len(y_test_t)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}")

    # -------------------------
    # 🔥 Early Stopping
    # -------------------------
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early Stopping triggered")
            break

Epoch 01/200 | Train Acc: 0.2080 | Test Acc: 0.1985 | Test Loss: 1.6098
Epoch 02/200 | Train Acc: 0.2076 | Test Acc: 0.1987 | Test Loss: 1.6098
Epoch 03/200 | Train Acc: 0.2076 | Test Acc: 0.1987 | Test Loss: 1.6098
Epoch 04/200 | Train Acc: 0.2077 | Test Acc: 0.1983 | Test Loss: 1.6098
Epoch 05/200 | Train Acc: 0.2078 | Test Acc: 0.1984 | Test Loss: 1.6098
Epoch 06/200 | Train Acc: 0.2077 | Test Acc: 0.1984 | Test Loss: 1.6098
Epoch 07/200 | Train Acc: 0.2077 | Test Acc: 0.1986 | Test Loss: 1.6098
Epoch 08/200 | Train Acc: 0.2076 | Test Acc: 0.1991 | Test Loss: 1.6098
Epoch 09/200 | Train Acc: 0.2076 | Test Acc: 0.1991 | Test Loss: 1.6098
Epoch 10/200 | Train Acc: 0.2076 | Test Acc: 0.1991 | Test Loss: 1.6099
Epoch 11/200 | Train Acc: 0.2077 | Test Acc: 0.1989 | Test Loss: 1.6099
Epoch 12/200 | Train Acc: 0.2076 | Test Acc: 0.1988 | Test Loss: 1.6099
Epoch 13/200 | Train Acc: 0.2077 | Test Acc: 0.1988 | Test Loss: 1.6099
Epoch 14/200 | Train Acc: 0.2077 | Test Acc: 0.1987 | Test Loss:

KeyboardInterrupt: 

## 위 학습을 맥북의 GPU(MPS)를 이용해 돌려보기

In [17]:
import torch

# -------------------------
# Device 설정 (MPS)
# -------------------------
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# numpy → torch 변환
# -------------------------
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)

X_test_t  = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test, dtype=torch.long).to(device)

# 모델도 GPU로
model = model.to(device)

Using device: mps


In [ ]:
EPOCHS = 200
batch_size = 1024

patience = 100
best_test_loss = float('inf')
early_stop_counter = 0

for epoch in range(EPOCHS):

    model.train()
    correct_train = 0
    epoch_loss = 0

    # -------------------------
    # Train
    # -------------------------
    for i in range(0, len(X_train_t), batch_size):

        xb = X_train_t[i:i+batch_size]
        yb = y_train_t[i:i+batch_size]

        optimizer.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        _, predicted_train = torch.max(outputs, 1)
        correct_train += (predicted_train == yb).sum().item()

    train_acc = correct_train / len(X_train_t)

    # -------------------------
    # Test
    # -------------------------
    model.eval()
    with torch.no_grad():

        outputs = model(X_test_t)
        test_loss = criterion(outputs, y_test_t)

        _, predicted_test = torch.max(outputs, 1)
        correct_test = (predicted_test == y_test_t).sum().item()

        test_acc = correct_test / len(y_test_t)

    print(f"Epoch {epoch+1:03d}/{EPOCHS} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}")

    # -------------------------
    # Early Stopping
    # -------------------------
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early Stopping triggered")
            break